# Bài 19: Document Upload
## Cho user tự mang tài liệu vào hệ thống

**Mục tiêu:**
- Dùng `st.file_uploader()` để nhận PDF từ user
- Đọc PDF từ **bộ nhớ** (bytes) thay vì từ đường dẫn đĩa
- Viết `services/ingest.py` — pipeline PDF → chunks → embeddings → ChromaDB
- Nối vào UI: user upload → hỏi đáp được ngay

**Tại sao tính năng này quan trọng:** đến giờ hệ thống chỉ trả lời được về 2 công ty bạn đã ingest sẵn. User thật không có PDF của bạn. Cho họ upload tài liệu riêng → app thành *product* thật sự.

---
## Phần 1: Lý thuyết

### `st.file_uploader()` trả về gì?

```python
uploaded = st.file_uploader("Tải PDF", type="pdf", accept_multiple_files=True)
```

- Chưa upload → trả về `None` (hoặc `[]` nếu `accept_multiple_files=True`)
- Đã upload → trả về `UploadedFile` — một object nằm **trong bộ nhớ (RAM)**, KHÔNG phải file trên đĩa
- Lấy nội dung: `uploaded.getvalue()` → trả về `bytes`
- Lấy tên file: `uploaded.name` → ví dụ `"TSLA-10Q.pdf"`

---

### Điểm khác biệt mấu chốt: đọc từ bytes, không phải path

Ở bài 13 bạn mở PDF từ **đường dẫn đĩa**:
```python
doc = fitz.open(pdf_path)                          # đọc từ file trên đĩa
```

File user upload nằm trong RAM, không có đường dẫn. Nên phải mở từ **stream bytes**:
```python
doc = fitz.open(stream=file_bytes, filetype="pdf")  # đọc từ bộ nhớ
```

Sau khi mở, mọi thứ còn lại (`.get_text()`, chunk, embed) giống hệt bài 13.

---

### User upload cho công ty nào? → phải gắn nhãn

Nhớ lại bài 13: query dùng `where={"symbol": symbol}`. Nếu upload mà không gắn `symbol`, chunk sẽ không bao giờ được tìm thấy khi hỏi. Nên khi upload, ta yêu cầu user nhập **mã công ty** (ví dụ `TSLA`) để gắn vào metadata:

```python
metadatas = [{"symbol": "TSLA", "source": "TSLA-10Q.pdf"} for _ in chunks]
```

> Chú ý ta thêm cả `"source"` (tên file) vào metadata — bài 20 (Citations) sẽ cần nó để hiện *"nguồn: file nào"*.

---

### Vị trí trong kiến trúc

```
main.py (UI)          st.file_uploader → gọi ingest
     ↓
services/ingest.py    bytes → text → chunks → embeddings → ChromaDB
     ↓
services/rag.py       (đã có) get_resources() → collection dùng chung
```

`ingest.py` sẽ dùng lại `get_resources()` từ `rag.py` để lấy đúng `embed_model` và `collection` — không tạo mới, dùng chung cái đã cache.

---
## Phần 2: Ví dụ — đọc PDF từ bytes

Chạy ô dưới để thấy `fitz.open(stream=...)` hoạt động. Ta đọc 1 file có sẵn thành bytes (giả lập file user upload), rồi mở từ bytes.

In [4]:
import fitz  # PyMuPDF

# Giả lập: đọc 1 PDF thành bytes (giống thứ st.file_uploader trả về)
with open(r"../phase4/data/NVDA-F1Q26-10-Q.pdf", "rb") as f:
    file_bytes = f.read()

print(f"Kiểu dữ liệu: {type(file_bytes)}")   # <class 'bytes'>
print(f"Kích thước: {len(file_bytes):,} bytes")

# Mở PDF TỪ BYTES (không phải từ path)
doc = fitz.open(stream=file_bytes, filetype="pdf")
print(f"Số trang: {doc.page_count}")
print(f"200 ký tự đầu trang 1:\n{doc[0].get_text()[:200]}")
doc.close()

Kiểu dữ liệu: <class 'bytes'>
Kích thước: 375,056 bytes
Số trang: 43
200 ký tự đầu trang 1:
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-Q
☒
QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the quarterly period end


Thấy rồi: chỉ cần đổi `fitz.open(path)` → `fitz.open(stream=bytes, filetype="pdf")`, phần còn lại y hệt bài 13.

---
## Phần 3: Bài tập

### Bước 1: Tạo `app/services/ingest.py`

Dưới đây là khung — điền các chỗ `TODO`. Tái sử dụng `split_into_chunks` từ bài 13.

```python
import fitz
from logging_setup import logger
from services.rag import get_resources


def split_into_chunks(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    # copy nguyên từ bài 13
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


def ingest_pdf(file_bytes: bytes, symbol: str, source_name: str) -> int:
    """Đọc PDF từ bytes, chunk, embed, lưu vào ChromaDB.
    Trả về số chunks đã thêm."""
    logger.info(f"[ingest] Bắt đầu ingest '{source_name}' cho symbol {symbol}")

    # 1. Mở PDF từ bytes và lấy toàn bộ text
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    full_text = ""
    for page_num in range(doc.page_count):
        full_text += doc[page_num].get_text()
    doc.close()

    # 2. Chunk
    chunks = split_into_chunks(full_text)

    # 3. Lấy embed_model + collection dùng chung (KHÔNG tạo mới)
    embed_model, collection = get_resources()

    # 4. TODO: embed chunks → list
    #    embeddings = embed_model.encode(___).tolist()

    # 5. TODO: tạo metadatas — mỗi chunk 1 dict, gồm CẢ symbol VÀ source
    #    metadatas = [{"symbol": ___, "source": ___} for _ in chunks]

    # 6. TODO: tạo ids unique — format gợi ý: f"{symbol}-{source_name}_chunk_{i}"
    #    (giống bài 13, dùng source_name để không trùng giữa các file)

    # 7. TODO: upsert vào collection (dùng upsert, KHÔNG add — để upload
    #    lại cùng file không bị lỗi 'ID already exists')
    #    collection.upsert(documents=___, embeddings=___, metadatas=___, ids=___)

    logger.info(f"[ingest] Đã thêm {len(chunks)} chunks từ '{source_name}'")
    return len(chunks)
```

---

### Bước 2: Nối vào sidebar trong `app/main.py`

Thêm vào trong khối `with st.sidebar:` (sau phần nhập mã cổ phiếu). Điền `TODO`:

```python
    st.markdown("---")
    st.subheader("📎 Tải lên báo cáo")

    upload_symbol = st.text_input(
        "Mã công ty cho tài liệu",
        help="Tài liệu sẽ được gắn nhãn công ty này để hỏi đáp"
    )
    uploaded_files = st.file_uploader(
        "Chọn PDF", type="pdf", accept_multiple_files=True
    )

    if st.button("➕ Thêm vào hệ thống"):
        # TODO: kiểm tra điều kiện trước khi ingest:
        #   - upload_symbol không được rỗng
        #   - uploaded_files phải có file
        # Nếu thiếu → st.warning(...) và không làm gì
        #
        # Nếu đủ → lặp qua từng file, gọi ingest_pdf(...) với:
        #   file.getvalue(), upload_symbol.upper().strip(), file.name
        # Rồi báo st.success(f"Đã thêm {n} chunks từ {file.name}")
        pass
```

Đầu file `main.py` nhớ thêm import:
```python
from services.ingest import ingest_pdf
```

---

### Bước 3: Test

1. `streamlit run app/main.py`
2. Ở sidebar, nhập mã (ví dụ `TSLA`), upload 1 PDF báo cáo Tesla bất kỳ, bấm **Thêm vào hệ thống**
3. Ở ô "Mã cổ phiếu" phía trên, gõ `TSLA`
4. Hỏi: *"Doanh thu của TSLA quý gần nhất?"*
5. Xem log terminal — phải thấy `[ingest] Đã thêm N chunks` và sau đó `[collect] TSLA: tìm thấy 3 chunks`

**Checklist:**
- [ ] Upload xong hiện thông báo success
- [ ] Hỏi về công ty vừa upload → trả lời dựa trên tài liệu đó
- [ ] Không cần restart app — data mới nhìn thấy ngay (vì `collection` là handle sống tới ChromaDB)

---

### Câu hỏi suy ngẫm (không bắt buộc code)

Sau khi upload, tại sao không cần gọi `get_resources.clear()` để làm mới cache? (Gợi ý: `collection` trỏ tới đâu, và `upsert` ghi vào đâu?)